In [ ]:
df = master.copy()

In [ ]:
#Part A-Train MLP Properly

##############################
#1.)
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
#######################################

########################################
#2.)
import os, random
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
########################################

#--------------------------------------
# 1.) Build y from one-hot cols
#----------------------------------------
#Goal: Turn 4 one-hot target columns into a single multi-class label

target_cols = [
    "study_group_healthy",
    "study_group_pre_diabetes_lifestyle_controlled",
    "study_group_oral_medication_and_or_non_insulin_injectable_medication_controlled",
    "study_group_insulin_dependent"
]

df[target_cols] = df[target_cols].astype(int)
print(df[target_cols].sum(axis=1).value_counts())

class_names = ["healthy", "prediabetes", "oral_medicated", "insulin_dependent"]

row_sums = df[target_cols].sum(axis=1)
print(row_sums.value_counts())
print("Bad rows (sum!=1):", (row_sums != 1).sum())

##########################

print("just before split section df columns (n):", df.shape[1])
print(df.columns.tolist())

#---------------------------
# 2.) Train/Val/Test Split use provided splits
#---------------------------
#Goal: Build feature matrices and class labels using the datasets pre made split columns

train_df = df[df["split_train"] == 1].copy()
val_df   = df[df["split_val"] == 1].copy()
test_df  = df[df["split_test"] == 1].copy()

#removing the target one-hot columns
drop_cols = target_cols + ["participant_id", "recommended_split", "split_train", "split_val", "split_test"]

#c is each column name, only include in the dataframe
X_train = train_df.drop(columns=[c for c in drop_cols if c in train_df.columns]).copy()
y_train = train_df[target_cols].values.argmax(axis=1)

# Make groups aligned with X_train rows
groups_train = train_df.loc[X_train.index, "participant_id"].to_numpy()

X_val = val_df.drop(columns=[c for c in drop_cols if c in val_df.columns]).copy()
y_val = val_df[target_cols].values.argmax(axis=1)

X_test = test_df.drop(columns=[c for c in drop_cols if c in test_df.columns]).copy()
y_test = test_df[target_cols].values.argmax(axis=1)

##############################

#----------------------------------
# 3.) Preprocessor
#------------------------------------
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print("X_train columns:", X_train.columns.tolist())
print("numeric_features (count):", len(numeric_features))
print("first numeric:", numeric_features[:20])
print("categorical_features (count):", len(categorical_features))
print("first categorical:", categorical_features[:20])

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, categorical_features),
    ],
    remainder="drop"
)

#Scales numeric columns, one-hot encodes categorical columns, handles missing values
X_train_p = preprocessor.fit_transform(X_train)
X_val_p   = preprocessor.transform(X_val)
X_test_p  = preprocessor.transform(X_test)

#densify if sparse
#checks if its a sparse matrix
X_train_p = X_train_p.toarray() if hasattr(X_train_p, "toarray") else X_train_p
X_val_p   = X_val_p.toarray()   if hasattr(X_val_p, "toarray")   else X_val_p
X_test_p  = X_test_p.toarray()  if hasattr(X_test_p, "toarray")  else X_test_p

# capped class weights
classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
final_cw = dict(zip(classes, weights))
final_cw = {k: float(min(v, 6.0)) for k, v in final_cw.items()}

print("Train y bincount:", np.bincount(y_train, minlength=4))
print("Train one-hot sums:", train_df[target_cols].sum().to_dict())
#########################


#-----------------------
# 4.) MLP Builder
#------------------------
#Build and train the MLP

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization

import tensorflow as tf

def sparse_focal_loss(gamma=4):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)

        # convert to one-hot
        y_true_onehot = tf.one_hot(y_true, depth=4)

        # numerical stability
        epsilon = 1e-7
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)

        # cross entropy
        ce = -y_true_onehot * tf.math.log(y_pred)

        # focal scaling
        weight = tf.pow(1 - y_pred, gamma)

        fl = weight * ce

        return tf.reduce_sum(fl, axis=1)

    return loss

def build_mlp(n_in, n_out=4, lr=1e-3):
    model = Sequential([
        Input(shape=(n_in,)),
        Dense(256, activation="relu"),
        BatchNormalization(),
        Dropout(0.3),

        Dense(128, activation="relu"),
        BatchNormalization(),
        Dropout(0.3),

        Dense(64, activation="relu"),
        BatchNormalization(),
        Dropout(0.2),

        Dense(n_out, activation="softmax")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=sparse_focal_loss(gamma=2.5),
        metrics=["accuracy"]
    )

    return model

mlp = build_mlp(n_in=X_train_p.shape[1], n_out=4)

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor="val_loss", mode="min",
    patience=25, restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss", factor=0.5,
    patience=5, min_lr=1e-5
)

mlp.fit(
    X_train_p, y_train,
    validation_data=(X_val_p, y_val),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)
###########################


#--------------------------
# 5.) GroupKFold CV
#--------------------------
#Goal: Estimate performance robustly while precenting leakage between related samples

gkf = GroupKFold(n_splits=5)

fold_acc, fold_f1 = [], []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_train, y_train, groups=groups_train), start=1):

    X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
    y_tr, y_va = y_train[tr_idx], y_train[va_idx]

    # Fit preprocessor on X_tr only
    X_tr_p = preprocessor.fit_transform(X_tr)
    X_va_p = preprocessor.transform(X_va)

    X_tr_p = X_tr_p.toarray() if hasattr(X_tr_p, "toarray") else X_tr_p
    X_va_p = X_va_p.toarray() if hasattr(X_va_p, "toarray") else X_va_p

    classes = np.unique(y_tr)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_tr)
    final_cw = dict(zip(classes, weights))

    # cap weights
    final_cw = {k: float(min(v, 4.5)) for k, v in final_cw.items()}
    print("class_weight:", final_cw)

    mlp = build_mlp(n_in=X_tr_p.shape[1], n_out=4)
    early_stop = EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True)

    mlp.fit(X_tr_p, y_tr, class_weight=final_cw, validation_data=(X_va_p, y_va), epochs=200, batch_size=32,
            callbacks=[early_stop, reduce_lr], verbose=0)

    probs = mlp.predict(X_va_p, verbose=0)

    for t in [0.20, 0.25, 0.30, 0.35]:
        y_pred = np.argmax(probs, axis=1)
        y_pred[probs[:, 3] > t] = 3

        from sklearn.metrics import recall_score
        recall_ins = recall_score(y_va, y_pred, labels=[3], average=None)[0]
        print(f"Threshold {t} → Insulin Recall: {recall_ins:.3f}")

    # pick ONE threshold to score fold metrics (example: fixed at 0.25)
    t = 0.3
    y_pred = np.argmax(probs, axis=1)
    y_pred[probs[:, 3] > t] = 3

    fold_acc.append(accuracy_score(y_va, y_pred))
    fold_f1.append(f1_score(y_va, y_pred, average="weighted"))

    from sklearn.metrics import precision_score, recall_score, f1_score

    probs = mlp.predict(X_va_p, verbose=0)

    for t in [0.20, 0.25, 0.30, 0.35]:
        y_pred = np.argmax(probs, axis=1)
        y_pred[probs[:, 3] > t] = 3

        prec = precision_score(y_va, y_pred, labels=[3], average=None)[0]
        rec  = recall_score(y_va, y_pred, labels=[3], average=None)[0]
        f1   = f1_score(y_va, y_pred, labels=[3], average=None)[0]

        print(f"t={t:.2f}  insulin P={prec:.3f} R={rec:.3f} F1={f1:.3f}")


print("\n=== CV Summary (split_train only) ===")
print("CV Acc:", np.mean(fold_acc), "+/-", np.std(fold_acc))
print("CV F1 :", np.mean(fold_f1),  "+/-", np.std(fold_f1))
###############################


#------------------------
# 6.) Final Mode Training
#-------------------------
#Train one final model on TRAIN, tune/early-stop on VAL, then evaluate on TEST

X_train_p = preprocessor.fit_transform(X_train)
X_val_p   = preprocessor.transform(X_val)
X_test_p  = preprocessor.transform(X_test)

X_train_p = X_train_p.toarray() if hasattr(X_train_p, "toarray") else X_train_p
X_val_p   = X_val_p.toarray() if hasattr(X_val_p, "toarray") else X_val_p
X_test_p  = X_test_p.toarray() if hasattr(X_test_p, "toarray") else X_test_p

# capped class weights
classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
final_cw = dict(zip(classes, weights))
final_cw = {k: float(min(v, 6.0)) for k, v in final_cw.items()}


#final_mlp = build_mlp(n_in=X_train_p.shape[1], n_out=4, seed=123)
final_mlp = build_mlp(n_in=X_train_p.shape[1], n_out=4)

early_stop = EarlyStopping(monitor="val_accuracy", mode="max", patience=15, restore_best_weights=True)
reduce_lr  = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5)

final_mlp.fit(
    X_train_p, y_train,
    validation_data=(X_val_p, y_val),
    epochs=200,
    batch_size=32,
    callbacks=[early_stop, reduce_lr],
    class_weight=final_cw,
    verbose=0
)

y_test_pred = final_mlp.predict(X_test_p, verbose=0).argmax(axis=1)
print("Pred counts:", np.bincount(y_test_pred, minlength=4))
print("True counts:", np.bincount(y_test, minlength=4))

# ---------------------------
# 6.1 ) Extra of saving
# ---------------------------
import joblib
import os

os.makedirs("artifacts", exist_ok=True)

# 1) Save the fitted preprocessor (VERY IMPORTANT)
joblib.dump(preprocessor, "artifacts/preprocessor.joblib")

# 2) Save class_names (optional, but nice)
joblib.dump(class_names, "artifacts/class_names.joblib")

# 3) Save model weights
final_mlp.save_weights("artifacts/final_mlp.weights.h5")

print("Saved: preprocessor + model weights to artifacts/")

# ----------------------------
# 7) Held-out test evaluation
# ----------------------------
#Goal: Evaluate once on TEST (data the model never saw during training or early stopping)

y_test_pred = final_mlp.predict(X_test_p, verbose=0).argmax(axis=1)

test_acc = accuracy_score(y_test, y_test_pred)
test_f1  = f1_score(y_test, y_test_pred, average="weighted")

print("\n=== Final Held-out Test Results ===")
print(f"Test Accuracy: {test_acc:.3f}")
print(f"Test W-F1:     {test_f1:.3f}")
print(classification_report(y_test, y_test_pred, target_names=class_names, digits=3))

cm = confusion_matrix(y_test, y_test_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(values_format="d", xticks_rotation=45)
plt.title("Confusion Matrix (Test)")
plt.show()
##################################


In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

pca = PCA(n_components=2)
X_reduced = pca.fit_transform(X_train_p)

plt.figure(figsize=(8,6))

scatter = plt.scatter(
    X_reduced[:, 0],
    X_reduced[:, 1],
    c=y_train,
    cmap="viridis",
    alpha=0.7
)

# Create legend entries manually
class_names = ["healthy", "prediabetes", "oral_medicated", "insulin_dependent"]
handles = []

for i in range(len(class_names)):
    handles.append(
        plt.Line2D(
            [0], [0],
            marker='o',
            color='w',
            label=class_names[i],
            markerfacecolor=scatter.cmap(scatter.norm(i)),
            markersize=8
        )
    )

plt.legend(handles=handles, title="Class")
plt.title("PCA Projection")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

In [ ]:
!pip install shap

In [ ]:
import numpy as np
import shap

# feature names after preprocessing
feature_names = [str(f) for f in preprocessor.get_feature_names_out()]

# make sure arrays are dense numpy arrays
X_train_shap = X_train_p.toarray() if hasattr(X_train_p, "toarray") else np.asarray(X_train_p)
X_test_shap  = X_test_p.toarray() if hasattr(X_test_p, "toarray") else np.asarray(X_test_p)

# smaller samples for speed
bg_size = min(100, len(X_train_shap))
test_size = min(200, len(X_test_shap))

rng = np.random.default_rng(0)
bg_idx = rng.choice(len(X_train_shap), size=bg_size, replace=False)
test_idx = rng.choice(len(X_test_shap), size=test_size, replace=False)

X_background = X_train_shap[bg_idx]
X_explain = X_test_shap[test_idx]

# build explainer
explainer = shap.Explainer(final_mlp, X_background)
shap_values = explainer(X_explain)

print("SHAP values shape:", shap_values.values.shape)

# print top 20 features per class
for k, cname in enumerate(class_names):
    class_shap = shap_values.values[:, :, k]
    mean_abs_shap = np.abs(class_shap).mean(axis=0)
    top_idx = np.argsort(mean_abs_shap)[::-1][:20]

    print(f"\nClass: {cname}")
    for i in top_idx:
        print(f"{feature_names[i]}: {mean_abs_shap[i]:.5f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

mean_abs_by_class = np.abs(shap_values.values).mean(axis=0)

# total importance across all classes
total_importance = mean_abs_by_class.sum(axis=1)

# top N most important features overall
top_n = 10
top_idx = np.argsort(total_importance)[::-1][:top_n]

# subset and clean labels
plot_values = mean_abs_by_class[top_idx]   # shape: (top_n, n_classes)
plot_features = [feature_names[i] for i in top_idx]
plot_features = [f.replace("num__", "").replace("cat__", "")[:40] for f in plot_features]

# reverse so biggest feature is at top
plot_values = plot_values[::-1]
plot_features = plot_features[::-1]

plt.style.use("ggplot")
fig, ax = plt.subplots(figsize=(9, 5.5))

left = np.zeros(len(plot_features))

for class_idx, cname in enumerate(class_names):
    vals = plot_values[:, class_idx]
    ax.barh(plot_features, vals, left=left, label=cname)
    left += vals

ax.set_xlabel("mean(|SHAP value|) (average impact on model output magnitude)")
ax.set_title("Top SHAP Features by Class")
ax.legend(title="Class", bbox_to_anchor=(1.02, 0.5), loc="center left")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score

feature_names = [str(f) for f in preprocessor.get_feature_names_out()]

def class_one_vs_rest_accuracy(y_true, y_pred, class_idx):
    return accuracy_score((y_true == class_idx).astype(int), (y_pred == class_idx).astype(int))

def permutation_importance_per_class(model, X, y, class_idx, n_repeats=3, random_state=0):
    rng = np.random.default_rng(random_state)

    base_prob = model.predict(X, verbose=0)
    base_pred = base_prob.argmax(axis=1)
    base_score = class_one_vs_rest_accuracy(y, base_pred, class_idx)

    importances = np.zeros(X.shape[1], dtype=float)

    for j in range(X.shape[1]):
        drops = []
        for _ in range(n_repeats):
            Xp = X.copy()
            rng.shuffle(Xp[:, j])
            pred = model.predict(Xp, verbose=0).argmax(axis=1)
            drops.append(base_score - class_one_vs_rest_accuracy(y, pred, class_idx))
        importances[j] = np.mean(drops)

    return base_score, importances

for k, cname in enumerate(class_names):
    base, imp = permutation_importance_per_class(final_mlp, X_test_p, y_test, class_idx=k, n_repeats=3)
    top_idx = np.argsort(imp)[::-1][:20]

    print(f"\nClass: {cname} | baseline one-vs-rest acc={base:.3f}")
    for i in top_idx:
        print(f"{feature_names[i]}: {imp[i]:.5f}")
